# PyDESeq2 - pseudobulk differential expression by clinical outcome (Good vs Poor)

- Reads the master object `outputs/h5ad_files/inprogress_3.h5ad` **strictly read-only** (`h5py.File(..., 'r')` plus `read_elem` on explicit subgroups). `X` (dense float64, ~7.6 GB) and `obsp` (two 95k x 95k graphs) are **never** loaded, and `sc.read_h5ad` is never called. Nothing is ever written back to the h5ad.
- Pulls only `obs` (a few columns), `layers['counts']` (CSR float32 raw integer counts) and `var/_index` (HGNC gene symbols).
- **Pseudobulks the 95,441 cells into 24 samples** by summing raw counts within `obs['batch']`. The sample is the pseudoreplication-safe unit of analysis: the model has **n = 24**, not n = 95,441. Cells from one patient are not independent replicates, so testing at cell level would inflate the effective sample size and the false-positive rate.
- Counts are cast to `int64` **before** the aggregation matmul: float32 is only exact up to 16,777,216 and the largest library is ~36.1M, so a float32 sum would silently round.
- Grouping label is `obs['clinical_outcome']` (13 Good / 11 Poor samples). `obs['batch_1']` is **stale** and is only used to demonstrate the mismatch - never to group or label.
- Design: `~clinical_outcome`, with `Poor` as the reference level so the design column reads `clinical_outcome[T.Good]`.
- **No covariates.** `obs['site']` is perfectly confounded with outcome, so adding it would make the design rank-deficient and remove the effect being tested. `site` is carried in the metadata for QC/plotting only.
- Contrast: `['clinical_outcome', 'Good', 'Poor']`. **Positive log2FoldChange = up in Good; negative = up in Poor.** This direction is asserted empirically from CPM group means, not assumed.
- Pre-filter: keep genes detected (count > 0) in at least 3 of the 24 pseudobulk samples. Significance: `padj < 0.05` **and** `|log2FoldChange| > 0.5`.
- Outputs: `DEG_results.csv`, `DEG_significant.csv` and `DEG_summary.md` in `outputs/csv_files/pydeseq2/clinical_outcome/`, plus five 600-dpi SVG QC figures in `outputs/images/pydeseq2/clinical_outcome/`.
- **Caveat - HVG universe.** This object carries a 10,000-gene highly-variable-gene panel, not the full transcriptome. Every statistic below (including the number of genes tested, the FDR correction and any downstream GSEA) is therefore relative to an HVG background, not to all ~20k protein-coding genes. Pathway results must be interpreted, and reported, against that restricted universe.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# necessary packages
import os
import logging
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

## Configuration

Every path is absolute. The h5ad path is listed for completeness but is only ever opened with `h5py` in mode `'r'`.

In [ ]:
# ---- paths (absolute) -------------------------------------------------------
# READ-ONLY master object. Opened with h5py mode 'r' only. Never sc.read_h5ad,
# never write_h5ad, never 'a'/'r+'.
H5AD_PATH = f'{ACC_DATA_ROOT}/outputs/h5ad_files/inprogress_3.h5ad'

CSV_DIR = f'{ACC_DATA_ROOT}/outputs/csv_files/pydeseq2/clinical_outcome'
IMG_DIR = f'{ACC_DATA_ROOT}/outputs/images/pydeseq2/clinical_outcome'

RESULTS_CSV = os.path.join(CSV_DIR, 'DEG_results.csv')
SIG_CSV     = os.path.join(CSV_DIR, 'DEG_significant.csv')
SUMMARY_MD  = os.path.join(CSV_DIR, 'DEG_summary.md')

FIG_DEPTH   = os.path.join(IMG_DIR, 'processed_pseudobulk_qc_sample_depth.svg')
FIG_PCA     = os.path.join(IMG_DIR, 'processed_pseudobulk_pca_clinical_outcome.svg')
FIG_DISP    = os.path.join(IMG_DIR, 'processed_pseudobulk_dispersion.svg')
FIG_MA      = os.path.join(IMG_DIR, 'processed_pseudobulk_ma_clinical_outcome.svg')
FIG_VOLCANO = os.path.join(IMG_DIR, 'processed_pseudobulk_volcano_clinical_outcome.svg')

os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(IMG_DIR, exist_ok=True)

# ---- verified facts, asserted rather than re-derived -------------------------
N_CELLS_EXPECTED  = 95441
N_GENES_EXPECTED  = 10000
NNZ_EXPECTED      = 93307416
N_SAMPLES_EXPECTED = 24
EXPECTED_PB_TOTAL = 390427539     # printed as a checksum, not a hard failure

# ---- analysis constants ------------------------------------------------------
SAMPLE_COL   = 'batch'            # already in A####### form, e.g. P23
GROUP_COL    = 'clinical_outcome' # Good / Poor - the ONLY grouping variable
MIN_CELLS_PER_SAMPLE  = 20
MIN_SAMPLES_EXPRESSED = 3         # gene must be detected in >= 3 of 24 samples
PADJ_CUTOFF = 0.05
LFC_CUTOFF  = 0.5
N_CPUS      = 8

# ---- project colour convention ----------------------------------------------
GOOD_COLOR = '#4DAF4A'
POOR_COLOR = '#E41A1C'
OUTCOME_COLORS = {'Good': GOOD_COLOR, 'Poor': POOR_COLOR}

# ---- fonts ------------------------------------------------------------------
# Arial is NOT installed here. Liberation Sans is, and is metrically identical to
# Arial (same advance widths). Listing both means:
#   * matplotlib resolves metrics from LiberationSans-Regular.ttf -> Arial-correct layout
#   * svg.fonttype='none' writes the stack 'Arial', 'Liberation Sans' into the SVG,
#     so Illustrator/journals with Arial render true Arial, others fall back cleanly.
# Setting font.family='Arial' alone would silently lay out with DejaVu Sans (~8% wider).
plt.rcParams['font.family'] = ['Arial', 'Liberation Sans']
plt.rcParams['svg.fonttype'] = 'none'      # keep SVG text editable in Illustrator/Inkscape

# 'Arial' is deliberately first so it lands in the SVG's font-family, but since it is not
# installed here font_manager logs findfont messages for EVERY text object. The lookup then
# falls through to Liberation Sans, which is exactly what we want, so the message is noise.
# (This is a logging call, not a warning, so warnings.filterwarnings does not catch it.)
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

# ---- publication font sizes -------------------------------------------------
plt.rcParams['font.size']             = 14
plt.rcParams['axes.titlesize']        = 16
plt.rcParams['axes.labelsize']        = 15
plt.rcParams['xtick.labelsize']       = 13
plt.rcParams['ytick.labelsize']       = 13
plt.rcParams['legend.fontsize']       = 13
plt.rcParams['legend.title_fontsize'] = 13

print(f'h5ad (read-only) : {H5AD_PATH}')
print(f'csv out          : {CSV_DIR}')
print(f'figure out       : {IMG_DIR}')
print(f'font             : {plt.rcParams["font.family"]}')

## 1. Memory-safe load

`read_elem` is pointed at explicit subgroups (`obs`, `layers/counts`, `var/_index`). That is exactly what keeps `X` (dense float64, ~7.6 GB) and `obsp` (two 95k x 95k sparse graphs) untouched on disk - `sc.read_h5ad` would materialise all of them, which is why it is banned here.

In [ ]:
try:
    read_elem = ad.io.read_elem
except AttributeError:
    from anndata.experimental import read_elem

# mode 'r' == read-only handle. Only the three named subgroups are pulled into RAM;
# f['X'] and f['obsp'] are inspected for shape at most, never read.
with h5py.File(H5AD_PATH, 'r') as f:
    assert f['layers/counts'].attrs['encoding-type'] == 'csr_matrix', 'counts layer is not CSR'
    obs = read_elem(f['obs'])[['batch', 'clinical_outcome', 'site', 'annot_5', 'batch_1']]
    counts = read_elem(f['layers/counts'])
    genes = pd.Index(read_elem(f['var/_index']), name='gene')
    print('X on disk, never read:', f['X'].shape, f['X'].dtype)

print('obs           :', obs.shape)
print('counts        :', counts.shape, counts.dtype, type(counts).__name__)
print('counts nnz    :', counts.nnz)
print('genes         :', len(genes), '->', list(genes[:5]))

assert obs.shape[0] == N_CELLS_EXPECTED, obs.shape
assert counts.shape == (N_CELLS_EXPECTED, N_GENES_EXPECTED), counts.shape
assert counts.nnz == NNZ_EXPECTED, counts.nnz
assert len(genes) == N_GENES_EXPECTED
assert obs[SAMPLE_COL].nunique() == N_SAMPLES_EXPECTED
print('\nOK - shape, dtype, nnz and sample count all match the verified facts')

## 2. Why `batch_1` is not used

`obs['batch_1']` carries a `good_` / `poor_` prefix from an earlier round of labelling. It is stale: at least one sample's prefix disagrees with its current `clinical_outcome`. The cell below shows the disagreement explicitly.

In [ ]:
b1 = obs['batch_1'].astype(str)
outcome_str = obs[GROUP_COL].astype(str)

# outcome implied by the batch_1 prefix, e.g. 'poor_P07' -> 'Poor'
implied = b1.str.extract(r'^(good|poor)_', expand=False).str.capitalize()
mismatch = implied.notna() & (implied != outcome_str)

print(f'cells whose batch_1 prefix disagrees with clinical_outcome: {int(mismatch.sum()):,}')
if mismatch.any():
    print('\ncrosstab restricted to the disagreeing rows (batch_1 x clinical_outcome):')
    print(pd.crosstab(b1[mismatch], outcome_str[mismatch]))
    offenders = (obs.loc[mismatch, ['batch', 'batch_1']]
                 .astype(str)
                 .assign(clinical_outcome=outcome_str[mismatch])
                 .drop_duplicates()
                 .reset_index(drop=True))
    print('\noffending sample(s):')
    print(offenders)
else:
    print('no disagreement found in this object')

print('\nbatch_1 is STALE: its prefix contradicts the current clinical annotation, so every')
print('grouping, label and colour in this notebook is taken from obs["clinical_outcome"] only.')

## 3. Pseudobulk: 95,441 cells -> 24 samples

Raw counts are summed within `obs['batch']` using a sparse indicator matrix `A` (samples x cells), so the aggregation is a single sparse matmul.

**The `int64` cast is not cosmetic.** `layers['counts']` is float32, which represents integers exactly only up to 16,777,216. The largest per-sample library here is ~36.1M, so summing in float32 would silently round the biggest samples. Casting the count matrix to `int64` *before* the matmul keeps every sum exact.

In [ ]:
def pseudobulk_by_sample(counts, sample_ids, gene_index, min_cells_per_sample=MIN_CELLS_PER_SAMPLE):
    '''Sum raw single-cell counts within each sample.

    Adapted from pseudobulk_by_sample() in sc_PyDESeq2_input_prep.ipynb, but working on the
    raw CSR matrix + a label vector instead of an AnnData, so the master h5ad is never opened
    as an AnnData at all.

    Returns (pb, cell_counts):
      pb          : samples x genes int64 DataFrame, index sorted sample IDs
      cell_counts : cells per retained sample
    '''
    if not sparse.issparse(counts):
        counts = sparse.csr_matrix(counts)

    ids = pd.Series(np.asarray(sample_ids).astype(str))

    cell_counts = ids.value_counts()
    keep_samples = cell_counts[cell_counts >= min_cells_per_sample].index
    keep_mask = ids.isin(keep_samples).to_numpy()
    n_dropped_cells = int((~keep_mask).sum())
    if n_dropped_cells:
        print(f'dropping {n_dropped_cells:,} cells from samples with < {min_cells_per_sample} cells')

    ids_f = ids[keep_mask]
    X_f = counts if keep_mask.all() else counts[keep_mask, :]

    # aggregation matrix A: [n_samples x n_cells], one 1 per cell
    uniq = pd.Index(sorted(ids_f.unique()), name='sample_id')
    row = pd.Series(np.arange(len(uniq)), index=uniq).loc[ids_f].to_numpy()
    col = np.arange(ids_f.shape[0])
    data = np.ones(ids_f.shape[0], dtype=np.int64)
    A = sparse.csr_matrix((data, (row, col)), shape=(len(uniq), ids_f.shape[0]))

    # CRITICAL: cast to int64 BEFORE the matmul - float32 is exact only to 16,777,216
    # and the largest library here is ~36.1M.
    pbX = np.asarray((A @ X_f.astype(np.int64)).todense())

    pb = pd.DataFrame(pbX, index=uniq, columns=gene_index).astype(np.int64)
    return pb, cell_counts.loc[uniq]


pb, cell_counts = pseudobulk_by_sample(counts, obs[SAMPLE_COL].values, genes)

del counts   # ~1 GB of CSR buffers released before PyDESeq2 starts

print(pb.shape, pb.values.dtype)
pb.iloc[:5, :6]

In [ ]:
# sample IDs are already in A####### form; this is a defensive no-op guard in case an
# object with the old good_/poor_ prefixes is ever fed in.
pb.index = pb.index.str.replace(r'^(good_|poor_)', 'A', regex=True)
pb.index.name = 'sample_id'

assert pb.index.str.fullmatch(r'A\d+').all(), pb.index.tolist()
assert pb.index.is_unique
assert (pb.dtypes == np.int64).all(), 'pseudobulk must be int64'
assert pb.shape == (N_SAMPLES_EXPECTED, N_GENES_EXPECTED), pb.shape
assert (pb.values >= 0).all(), 'negative pseudobulk counts'

cell_counts.index = cell_counts.index.str.replace(r'^(good_|poor_)', 'A', regex=True)

total_counts = int(pb.values.sum())
largest_lib = int(pb.sum(axis=1).max())
print(f'samples x genes      : {pb.shape[0]} x {pb.shape[1]}')
print(f'pseudobulk total sum : {total_counts:,}')
print(f'expected checksum    : {EXPECTED_PB_TOTAL:,}')
print('checksum             :', 'PASS' if total_counts == EXPECTED_PB_TOTAL else 'FAIL')
print(f'largest library      : {largest_lib:,}   '
      f'(float32 exact-integer limit is 16,777,216 - '
      f'{"exceeded, so the int64 cast was required" if largest_lib > 16777216 else "not exceeded here"})')

## 4. Sample-level metadata

One row per sample. `clinical_outcome` is made categorical with `Poor` first, so `Poor` is the reference level and the design column reads `clinical_outcome[T.Good]`.

`site` is stored for QC and figure shapes only - it is **not** in the design, because it is perfectly confounded with outcome.

In [ ]:
obs_s = obs.assign(**{SAMPLE_COL: obs[SAMPLE_COL].astype(str)})

# each sample must carry exactly one outcome / site value before taking the mode
nun = obs_s.groupby(SAMPLE_COL, observed=True)[[GROUP_COL, 'site']].agg(lambda s: s.dropna().nunique())
assert (nun[GROUP_COL] == 1).all(), nun[nun[GROUP_COL] != 1]
assert (nun['site'] == 1).all(), nun[nun['site'] != 1]

def mode_or_nan(x):
    x = x.dropna()
    return np.nan if x.empty else x.value_counts().index[0]

labels = obs_s.groupby(SAMPLE_COL, observed=True)[GROUP_COL].agg(mode_or_nan).astype(str)
sites  = obs_s.groupby(SAMPLE_COL, observed=True)['site'].agg(mode_or_nan).astype(str)
labels.index = labels.index.str.replace(r'^(good_|poor_)', 'A', regex=True)
sites.index  = sites.index.str.replace(r'^(good_|poor_)', 'A', regex=True)

metadata = pd.DataFrame(index=pb.index)
# Poor listed first -> reference level -> design column is clinical_outcome[T.Good]
metadata[GROUP_COL]      = pd.Categorical(labels.loc[pb.index].values, categories=['Poor', 'Good'])
metadata['n_cells']        = cell_counts.loc[pb.index].values
metadata['library_size']   = pb.sum(axis=1).values
metadata['detected_genes'] = (pb > 0).sum(axis=1).values
metadata['site']           = sites.loc[pb.index].values   # QC ONLY - never in the design

vc = metadata[GROUP_COL].value_counts()
assert int(vc['Good']) == 13 and int(vc['Poor']) == 11, vc.to_dict()
assert metadata[GROUP_COL].isna().sum() == 0

print('samples per outcome :', {'Good': int(vc['Good']), 'Poor': int(vc['Poor'])})
print('reference level     :', metadata[GROUP_COL].cat.categories[0])
print('\nsite x clinical_outcome - perfectly confounded, hence no covariates in the design:')
print(pd.crosstab(metadata['site'], metadata[GROUP_COL]))
metadata

## 5. Pre-filter

Rule: **keep a gene if it has a non-zero count in at least 3 of the 24 pseudobulk samples.**

Why not the usual DESeq2-vignette filter (e.g. `rowSums(counts) >= 10` or a CPM-based cut)?

- This object is **already HVG-filtered upstream** to 10,000 genes, so the low-expression tail the vignette filter is designed to remove has largely been removed already.
- Applying the vignette filter here would drop roughly 29% of the panel. Those genes are testable, and independent filtering inside `DeseqStats` (`independent_filter=True`) already optimises the FDR by discarding low-count genes at the p-value stage - so a hard pre-filter buys no extra power.
- It would also **shrink the GSEA background**: the ranked list handed to downstream preranked GSEA is exactly the set of tested genes, so throwing away a third of the panel changes every enrichment score for no statistical gain.
- The 3-sample rule only removes genes that are *structurally untestable* (present in fewer samples than the smallest group can support), which is the minimum needed to keep dispersion estimation well-posed.

In [ ]:
keep = (pb > 0).sum(axis=0) >= MIN_SAMPLES_EXPRESSED
pb_f = pb.loc[:, keep]

n_total_genes = int(pb.shape[1])
n_tested      = int(pb_f.shape[1])
n_dropped     = n_total_genes - n_tested

print(f'genes in pseudobulk matrix : {n_total_genes:,}')
print(f'genes kept (detected in >= {MIN_SAMPLES_EXPRESSED} of {pb.shape[0]} samples) : {n_tested:,}')
print(f'genes dropped              : {n_dropped:,}  ({100 * n_dropped / n_total_genes:.2f}%)')
assert n_tested > 0

## 6. Differential expression

`design` is given as a **formula string** (`'~clinical_outcome'`). The older `design_factors=` argument is deprecated and is not used.

In [ ]:
dds = DeseqDataSet(
    counts=pb_f,
    metadata=metadata,
    design='~clinical_outcome',
    refit_cooks=True,
    n_cpus=N_CPUS,
)

dds.deseq2()

design_cols = dds.obsm['design_matrix'].columns.tolist()
print('\ndesign matrix columns:', design_cols)
assert design_cols == ['Intercept', 'clinical_outcome[T.Good]'], design_cols

In [ ]:
stat_res = DeseqStats(
    dds,
    contrast=['clinical_outcome', 'Good', 'Poor'],
    alpha=PADJ_CUTOFF,
    cooks_filter=True,
    independent_filter=True,
    n_cpus=N_CPUS,
)

stat_res.summary()

res = stat_res.results_df.copy()
res.index.name = 'gene'
print('\nresults:', res.shape, list(res.columns))
res.head()

**Direction convention.** The contrast is `Good` vs `Poor` with `Poor` as reference, so

- `log2FoldChange > 0` -> the gene is **higher in Good**
- `log2FoldChange < 0` -> the gene is **higher in Poor**

The next cell verifies this empirically rather than trusting the convention.

## 7. Empirical check of the fold-change direction

In [ ]:
# CPM straight from the pseudobulk counts - independent of anything PyDESeq2 computed
cpm = pb_f.div(pb_f.sum(axis=1), axis=0) * 1e6
good_samples = metadata.index[metadata[GROUP_COL].astype(str) == 'Good']
poor_samples = metadata.index[metadata[GROUP_COL].astype(str) == 'Poor']
print(f'{len(good_samples)} Good samples, {len(poor_samples)} Poor samples')

sig_mask = (res['padj'] < PADJ_CUTOFF) & (res['log2FoldChange'].abs() > LFC_CUTOFF)
check = res[sig_mask]
if check.shape[0] == 0:
    print('no gene passed both thresholds - falling back to padj < 0.05 for the direction check')
    check = res[res['padj'] < PADJ_CUTOFF]

# each direction is selected by the SIGN of the fold change, so a direction with no
# significant genes reports 'none' instead of pulling in genes of the opposite sign
top_pos = check[check['log2FoldChange'] > 0].sort_values('log2FoldChange', ascending=False).head(5)
top_neg = check[check['log2FoldChange'] < 0].sort_values('log2FoldChange', ascending=True).head(5)

print(f'\ntop {top_pos.shape[0]} positive log2FoldChange (expected: mean CPM Good > Poor)')
if top_pos.shape[0] == 0:
    print('  none')
for g in top_pos.index:
    mg, mp = float(cpm.loc[good_samples, g].mean()), float(cpm.loc[poor_samples, g].mean())
    print(f'  {g:<15s} log2FC={top_pos.loc[g, "log2FoldChange"]:+7.3f}  CPM Good={mg:11.2f}  CPM Poor={mp:11.2f}')
    assert mg > mp, f'{g}: positive log2FC but mean CPM Good ({mg}) <= Poor ({mp})'

print(f'\ntop {top_neg.shape[0]} negative log2FoldChange (expected: mean CPM Poor > Good)')
if top_neg.shape[0] == 0:
    print('  none')
for g in top_neg.index:
    mg, mp = float(cpm.loc[good_samples, g].mean()), float(cpm.loc[poor_samples, g].mean())
    print(f'  {g:<15s} log2FC={top_neg.loc[g, "log2FoldChange"]:+7.3f}  CPM Good={mg:11.2f}  CPM Poor={mp:11.2f}')
    assert mp > mg, f'{g}: negative log2FC but mean CPM Poor ({mp}) <= Good ({mg})'

n_checked = top_pos.shape[0] + top_neg.shape[0]
assert n_checked > 0, 'no gene available to verify the fold-change direction'
print(f'\ndirection OK: positive log2FoldChange = up in Good  ({n_checked} genes verified against CPM group means)')

## 8. Significance calls and CSV export

A gene is significant when `padj < 0.05` **and** `|log2FoldChange| > 0.5`. Genes whose `padj` is `NaN` (filtered out by Cook's or by independent filtering) evaluate to `False`, which is the correct behaviour - the `NaN`s are never filled or dropped, so the exported table keeps the full tested gene set for downstream ranking.

In [ ]:
res['significant'] = (res['padj'] < PADJ_CUTOFF) & (res['log2FoldChange'].abs() > LFC_CUTOFF)
sigs = res[res['significant']].sort_values('log2FoldChange', ascending=False)

n_sig     = int(res['significant'].sum())
n_up_good = int((sigs['log2FoldChange'] > 0).sum())
n_up_poor = int((sigs['log2FoldChange'] < 0).sum())
n_padj_na = int(res['padj'].isna().sum())

print(f'genes tested        : {res.shape[0]:,}')
print(f'padj = NaN          : {n_padj_na:,}')
print(f'significant         : {n_sig:,}   (padj < {PADJ_CUTOFF} and |log2FC| > {LFC_CUTOFF})')
print(f'  up in Good (LFC>0): {n_up_good:,}')
print(f'  up in Poor (LFC<0): {n_up_poor:,}')
assert n_up_good + n_up_poor == n_sig
sigs.head(10)

In [ ]:
# exported schema is exactly baseMean,log2FoldChange,lfcSE,stat,pvalue,padj with index 'gene'
SCHEMA = ['baseMean', 'log2FoldChange', 'lfcSE', 'stat', 'pvalue', 'padj']

res_out = res[SCHEMA].copy()
sig_out = sigs[SCHEMA].copy()
res_out.index.name = 'gene'
sig_out.index.name = 'gene'

res_out.to_csv(RESULTS_CSV)
sig_out.to_csv(SIG_CSV)

print(f'{RESULTS_CSV}  ->  {res_out.shape[0]:,} rows')
print(f'{SIG_CSV}  ->  {sig_out.shape[0]:,} rows')
print('columns:', list(res_out.columns))

## 9. Plain-text summary

Every number below is read out of the in-memory objects - nothing is hard-coded.

In [ ]:
n_good = int((metadata[GROUP_COL].astype(str) == 'Good').sum())
n_poor = int((metadata[GROUP_COL].astype(str) == 'Poor').sum())

lines = [
    '# Differential expression - clinical outcome (Good vs Poor)',
    '',
    '## Source',
    '',
    f'- Object: `{H5AD_PATH}`',
    '- Opened **read-only** with h5py (mode `r`); only `obs`, `layers/counts` and `var/_index` were read.',
    '  `X` and `obsp` were never loaded and the file was never modified.',
    f'- Raw counts: `layers["counts"]`, {obs.shape[0]:,} cells x {n_total_genes:,} genes.',
    '',
    '## Unit of analysis',
    '',
    f'- **{pb.shape[0]} pseudobulk samples** ({n_good} Good, {n_poor} Poor), not {obs.shape[0]:,} cells.',
    '  Counts were summed per sample (`obs["batch"]`), so cells are not treated as replicates.',
    f'- Pseudobulk total counts: {total_counts:,}.',
    '',
    '## Model',
    '',
    '- Design: `~clinical_outcome` (reference level = `Poor`; design column `clinical_outcome[T.Good]`).',
    '- Contrast: `clinical_outcome`, `Good` vs `Poor`.',
    '  **Positive log2FoldChange = higher in Good; negative = higher in Poor.**',
    '- Covariates: **none**. `site` is perfectly confounded with outcome, so including it would',
    '  absorb the effect being tested; it is kept for QC only.',
    f'- Software: PyDESeq2, `refit_cooks=True`, `cooks_filter=True`, `independent_filter=True`, alpha = {PADJ_CUTOFF}.',
    '',
    '## Statistics',
    '',
    f'- Genes in pseudobulk matrix: **{n_total_genes:,}**',
    f'- Genes tested after pre-filter: **{n_tested:,}** (rule: non-zero count in at least',
    f'  {MIN_SAMPLES_EXPRESSED} of {pb.shape[0]} samples; {n_dropped:,} dropped)',
    f'- Genes with `padj` = NA (Cook\'s / independent filtering): **{n_padj_na:,}**',
    f'- Significant (`padj` < {PADJ_CUTOFF} and |log2FC| > {LFC_CUTOFF}): **{n_sig:,}**',
    f'  - Up in Good: **{n_up_good:,}**',
    f'  - Up in Poor: **{n_up_poor:,}**',
    '',
    '## Files',
    '',
    f'- `{RESULTS_CSV}` - all tested genes',
    f'- `{SIG_CSV}` - significant genes only',
    f'- Figures: `{IMG_DIR}`',
    '',
    '## Caveat - HVG universe',
    '',
    f'- The object carries a {n_total_genes:,}-gene highly-variable-gene panel, **not the full',
    '  transcriptome**. The gene counts above, the multiple-testing correction and any downstream',
    '  GSEA are therefore relative to an HVG background and must be reported as such.',
    '',
]

with open(SUMMARY_MD, 'w') as fh:
    fh.write('\n'.join(lines) + '\n')

print(f'written: {SUMMARY_MD}\n')
print('\n'.join(lines))

## 10. QC figures

All figures are 600-dpi SVGs with editable text, coloured by the project convention: Good = `#4DAF4A`, Poor = `#E41A1C`.

In [ ]:
# ---- sample depth: cells per sample and pseudobulk library size --------------
order = metadata.sort_values('library_size', ascending=False).index
bar_colors = [OUTCOME_COLORS[o] for o in metadata.loc[order, GROUP_COL].astype(str)]
x = np.arange(len(order))

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].bar(x, metadata.loc[order, 'n_cells'].values, color=bar_colors, edgecolor='none')
axes[0].set_ylabel('cells per sample')
axes[0].set_title(f'Pseudobulk depth - {pb.shape[0]} samples (the unit of analysis)')

axes[1].bar(x, metadata.loc[order, 'library_size'].values / 1e6, color=bar_colors, edgecolor='none')
axes[1].set_ylabel('library size (million counts)')
axes[1].set_xlabel('sample')
axes[1].set_xticks(x)
axes[1].set_xticklabels(order, rotation=90)

handles = [mpatches.Patch(facecolor=GOOD_COLOR, label=f'Good (n={n_good})'),
           mpatches.Patch(facecolor=POOR_COLOR, label=f'Poor (n={n_poor})')]
axes[0].legend(handles=handles, title='clinical outcome', frameon=False, loc='upper right')
for a in axes:
    sns.despine(ax=a)
fig.tight_layout()
fig.savefig(FIG_DEPTH, format='svg', dpi=600, bbox_inches='tight')
plt.show()
print('written:', FIG_DEPTH)

In [ ]:
# ---- PCA of VST pseudobulk ---------------------------------------------------
# VST is fitted on a SEPARATE clone so the DE object (dds) is never mutated by vst().
dds_vst = DeseqDataSet(
    counts=pb_f,
    metadata=metadata,
    design='~clinical_outcome',
    n_cpus=N_CPUS,
    quiet=True,
)
dds_vst.fit_size_factors()
dds_vst.vst(use_design=False)          # blind VST - design not used to fit dispersions
vst_mat = np.asarray(dds_vst.layers['vst_counts'])
print('vst matrix:', vst_mat.shape)

# drop zero-variance genes, centre, then SVD
v = vst_mat[:, vst_mat.std(axis=0) > 0]
vc_ = v - v.mean(axis=0, keepdims=True)
U, S, Vt = np.linalg.svd(vc_, full_matrices=False)
pcs = U * S
var_ratio = (S ** 2) / (S ** 2).sum() * 100

sites_u = sorted(metadata['site'].astype(str).unique())
marker_cycle = ['o', 's', '^', 'D', 'v', 'P', 'X', '*']
site_markers = {s: marker_cycle[i % len(marker_cycle)] for i, s in enumerate(sites_u)}

fig, ax = plt.subplots(figsize=(8, 7))
for s in sites_u:
    for o in ['Good', 'Poor']:
        m = (metadata['site'].astype(str) == s) & (metadata[GROUP_COL].astype(str) == o)
        if not m.any():
            continue
        idx = np.where(m.values)[0]
        ax.scatter(pcs[idx, 0], pcs[idx, 1], c=OUTCOME_COLORS[o], marker=site_markers[s],
                   s=110, edgecolor='black', linewidth=0.6, zorder=3)

ax.set_xlabel(f'PC1 ({var_ratio[0]:.1f}% variance)')
ax.set_ylabel(f'PC2 ({var_ratio[1]:.1f}% variance)')
ax.set_title('VST pseudobulk PCA - colour = outcome, shape = site')

h_outcome = [mpatches.Patch(facecolor=OUTCOME_COLORS[o], label=o) for o in ['Good', 'Poor']]
h_site = [Line2D([0], [0], marker=site_markers[s], color='none', markerfacecolor='lightgrey',
                 markeredgecolor='black', markersize=10, label=s) for s in sites_u]
leg1 = ax.legend(handles=h_outcome, title='clinical outcome', frameon=False,
                 loc='upper left', bbox_to_anchor=(1.02, 1.0))
ax.add_artist(leg1)
ax.legend(handles=h_site, title='site (confounded)', frameon=False,
          loc='upper left', bbox_to_anchor=(1.02, 0.55))
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(FIG_PCA, format='svg', dpi=600, bbox_inches='tight')
plt.show()
print('written:', FIG_PCA)
print('site is perfectly confounded with outcome - the shapes document why no covariate is modelled')

In [ ]:
# ---- dispersion plot ---------------------------------------------------------
# plot_dispersions() builds its own figure and calls plt.show() internally, which detaches the
# figure before we can save it with the project's settings. _hold_figure() suppresses that
# internal show() for the duration of the call, so plt.gcf() still holds the real axes and the
# figure is saved by us -> bbox_inches='tight' and the ['Arial', 'Liberation Sans'] font stack
# apply here exactly as they do to the other figures. save_path is deliberately NOT passed.
# rasterized=True is forwarded to plt.scatter and keeps the ~3 x n_genes point cloud as one
# embedded raster layer instead of tens of thousands of vector circles (MBs -> KBs).
from contextlib import contextmanager


@contextmanager
def _hold_figure():
    """Suppress a library's internal plt.show() so the figure survives for our own savefig."""
    _show = plt.show
    plt.show = lambda *args, **kwargs: None
    try:
        yield
    finally:
        plt.show = _show


try:
    with _hold_figure():
        dds.plot_dispersions(rasterized=True)
    fig = plt.gcf()
    assert fig.get_axes(), 'plot_dispersions returned no axes'
    fig.savefig(FIG_DISP, format='svg', dpi=600, bbox_inches='tight')
    plt.show()
    print('written:', FIG_DISP)
except Exception as exc:
    print(f'WARNING: plot_dispersions failed ({type(exc).__name__}: {exc}) - figure skipped')
finally:
    # make_scatter() mutates rcParams['font.size'] globally; restore the publication size
    plt.rcParams['font.size'] = 14


In [ ]:
# ---- MA plot -----------------------------------------------------------------
# Same handling as the dispersion plot: hold the figure, then save it ourselves.
try:
    with _hold_figure():
        stat_res.plot_MA(s=6, rasterized=True)
    fig = plt.gcf()
    assert fig.get_axes(), 'plot_MA returned no axes'
    fig.savefig(FIG_MA, format='svg', dpi=600, bbox_inches='tight')
    plt.show()
    print('written:', FIG_MA)
except Exception as exc:
    print(f'WARNING: plot_MA failed ({type(exc).__name__}: {exc}) - figure skipped')
finally:
    plt.rcParams['font.size'] = 14


In [ ]:
# ---- volcano -----------------------------------------------------------------
vol = res.dropna(subset=['padj', 'log2FoldChange']).copy()
vol['neglog10padj'] = -np.log10(vol['padj'].clip(lower=1e-300))

up_good = (vol['padj'] < PADJ_CUTOFF) & (vol['log2FoldChange'] > LFC_CUTOFF)
up_poor = (vol['padj'] < PADJ_CUTOFF) & (vol['log2FoldChange'] < -LFC_CUTOFF)
ns = ~(up_good | up_poor)

fig, ax = plt.subplots(figsize=(9, 8))
ax.scatter(vol.loc[ns, 'log2FoldChange'], vol.loc[ns, 'neglog10padj'],
           c='#BDBDBD', s=10, alpha=0.6, linewidth=0, rasterized=True, label='not significant')
ax.scatter(vol.loc[up_good, 'log2FoldChange'], vol.loc[up_good, 'neglog10padj'],
           c=GOOD_COLOR, s=16, alpha=0.9, linewidth=0, label=f'up in Good (n={int(up_good.sum())})')
ax.scatter(vol.loc[up_poor, 'log2FoldChange'], vol.loc[up_poor, 'neglog10padj'],
           c=POOR_COLOR, s=16, alpha=0.9, linewidth=0, label=f'up in Poor (n={int(up_poor.sum())})')

ax.axhline(-np.log10(PADJ_CUTOFF), color='black', linestyle='--', linewidth=1, alpha=0.7)
ax.axvline(LFC_CUTOFF, color='black', linestyle='--', linewidth=1, alpha=0.7)
ax.axvline(-LFC_CUTOFF, color='black', linestyle='--', linewidth=1, alpha=0.7)

# label the 10 most significant genes in each direction
for mask in (up_good, up_poor):
    top = vol[mask].sort_values('neglog10padj', ascending=False).head(10)
    for g, row in top.iterrows():
        ax.annotate(str(g), (row['log2FoldChange'], row['neglog10padj']),
                    fontsize=9, xytext=(3, 3), textcoords='offset points')

ax.set_xlabel('log2 fold change  (Good vs Poor)')
ax.set_ylabel('-log10 adjusted p-value')
ax.set_title(f'Pseudobulk DE - Good vs Poor (n={pb.shape[0]} samples)')
ax.legend(frameon=False, loc='upper left')
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(FIG_VOLCANO, format='svg', dpi=600, bbox_inches='tight')
plt.show()
print('written:', FIG_VOLCANO)

## 11. Completion report

In [ ]:
outputs = [RESULTS_CSV, SIG_CSV, SUMMARY_MD, FIG_DEPTH, FIG_PCA, FIG_DISP, FIG_MA, FIG_VOLCANO]

print('outputs written')
print('-' * 96)
for p in outputs:
    if os.path.exists(p):
        print(f'  {os.path.getsize(p) / 1024:10.1f} KB   {p}')
    else:
        print(f'  {"MISSING":>10s}      {p}')

print('\nkey counts')
print('-' * 96)
print(f'  cells read (never modelled directly) : {obs.shape[0]:,}')
print(f'  pseudobulk samples modelled          : {pb.shape[0]}  ({n_good} Good / {n_poor} Poor)')
print(f'  pseudobulk total counts              : {total_counts:,}  '
      f'({"PASS" if total_counts == EXPECTED_PB_TOTAL else "FAIL"} vs {EXPECTED_PB_TOTAL:,})')
print(f'  genes in matrix / tested / dropped   : {n_total_genes:,} / {n_tested:,} / {n_dropped:,}')
print(f'  padj = NaN                           : {n_padj_na:,}')
print(f'  significant (padj<{PADJ_CUTOFF}, |LFC|>{LFC_CUTOFF})   : {n_sig:,}')
print(f'    up in Good                         : {n_up_good:,}')
print(f'    up in Poor                         : {n_up_poor:,}')
print('\nThe master h5ad was opened read-only and was not modified.')